# 31 — Logistic Regression baseline

Baseline lineal escalable mediante **SGD con pérdida logística**, one-hot acotado y entrenamiento streaming sobre TRAIN completo. La evaluación permanece en VALIDATION; TEST sigue sellado.

In [ ]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m3_gate = paths.docs / "decisions" / "M3_GATE.md"
assert m3_gate.exists() and "M3 ✅ CLOSED" in m3_gate.read_text(encoding="utf-8"), (
    "M3 no está formalmente cerrado. No iniciar Modeling."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
MODEL_DIR = paths.root / "artifacts" / "models" / "m4"
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

{
    "gold_schema": GOLD_SPEC.schema_version,
    "modeling_schema": MODEL_SPEC.schema_version,
    "target": MODEL_SPEC.target,
    "model_matrix": MODEL_SPEC.model_matrix,
    "operational_review_fraction": MODEL_SPEC.ranking.operational_fraction,
}

assert (paths.reports / "modeling" / "30_validation.parquet").exists(), "Ejecuta 30 primero."
from argos.models.linear import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION


In [ ]:
from argos.models.linear import train_linear_model

result = train_linear_model(
    root=paths.root,
    gold_spec=GOLD_SPEC,
    modeling_spec=MODEL_SPEC,
    reports_root=paths.reports,
    model_dir=MODEL_DIR,
    manifest_dir=MANIFEST_DIR,
)
result.metrics


In [ ]:
result.ranking

In [ ]:
result.validation

In [ ]:
blocking = result.validation.loc[result.validation["blocking"]]
print("31 ✅ PASS" if result.ready_for_next_notebook else "31 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Model:", result.model_path)
assert result.ready_for_next_notebook
